# Streamlit â€” Personal Projects Guide

This notebook covers everything you need to build personal projects in Streamlit.
Each section explains a concept and shows a real code snippet you can copy directly.

**How Streamlit works:** Every time a user interacts with a widget (clicks a button, types in a box), the entire script reruns from top to bottom. Keep this mental model â€” it explains most of Streamlit's behaviour.

---
## Table of Contents
1. Setup & Page Config
2. Text Elements
3. Input Widgets
4. Layout (Columns, Tabs, Sidebar, Expander)
5. Displaying Data
6. Charts
7. Session State
8. Forms
9. Caching
10. File Upload & Download
11. Status & Feedback
12. Putting It Together â€” Mini Finance App

---
## 1. Setup & Page Config

`st.set_page_config` must be the **first Streamlit call** in your script â€” before any other `st.*` call.

In [ ]:
import streamlit as st

st.set_page_config(
    page_title="My App",       # browser tab title
    page_icon="ðŸ’°",             # emoji or path to .ico file
    layout="wide",             # 'centered' (default) or 'wide'
    initial_sidebar_state="expanded"  # 'expanded' or 'collapsed'
)

---
## 2. Text Elements

These render static text and markdown in the app.

In [ ]:
st.title("Main Title")                    # largest heading
st.header("Section Header")
st.subheader("Subsection")
st.text("Plain monospace text")           # fixed-width, no formatting
st.write("Flexible â€” handles strings, dataframes, dicts, charts")  # smart renderer

# Markdown â€” supports full GitHub-flavoured markdown
st.markdown("""
**Bold**, *italic*, `code`

- bullet one
- bullet two
""")

# Coloured callout boxes
st.success("Operation completed")
st.error("Something went wrong")
st.warning("Double-check this")
st.info("Just so you know")

# Metric cards â€” great for dashboards
st.metric(label="Total Spent", value="Â£340.50", delta="-Â£20.00")
# delta is green when positive, red when negative (flip with delta_color='inverse')

---
## 3. Input Widgets

Every widget returns a value. Assign it to a variable and use it in your logic.
When the user changes a widget, the script reruns and the variable gets the new value.

In [ ]:
# --- Text ---
name = st.text_input("Your name", placeholder="e.g. Zarif", value="")
notes = st.text_area("Notes", height=100)

# --- Numbers ---
budget = st.number_input("Monthly budget (Â£)", min_value=0.0, max_value=10000.0, value=500.0, step=50.0)
months = st.slider("Number of months", min_value=1, max_value=24, value=6)

# --- Selections ---
currency = st.selectbox("Currency", ["GBP", "USD", "EUR", "MYR"])
selected = st.multiselect("Categories to show", ["Food", "Transport", "Subscriptions"], default=["Food"])
view = st.radio("View", ["Monthly", "Weekly", "Daily"])

# --- Toggles ---
show_raw = st.checkbox("Show raw data", value=False)
dark_mode = st.toggle("Dark mode")  # same as checkbox but looks like a switch

# --- Date ---
import datetime
start_date = st.date_input("Start date", value=datetime.date.today())

# --- Button ---
# Buttons return True only on the rerun triggered by clicking them
if st.button("Calculate"):
    st.write(f"Budget for {months} months: Â£{budget * months:.2f}")

# Primary (blue) button
if st.button("Save", type="primary"):
    st.success("Saved!")

---
## 4. Layout

### 4a. Columns â€” side-by-side content

In [ ]:
# Equal columns
col1, col2, col3 = st.columns(3)
col1.metric("Income", "Â£1,200")
col2.metric("Expenses", "Â£800")
col3.metric("Savings", "Â£400", delta="+Â£50")

# Weighted columns â€” col1 is twice as wide as col2
col_wide, col_narrow = st.columns([2, 1])
with col_wide:
    st.write("Main content here")
with col_narrow:
    st.write("Sidebar-like info")

### 4b. Tabs

In [ ]:
tab1, tab2, tab3 = st.tabs(["Overview", "Transactions", "Settings"])

with tab1:
    st.write("Overview content")

with tab2:
    st.write("Transaction table here")

with tab3:
    st.write("Settings controls here")

### 4c. Sidebar

In [ ]:
# Anything inside st.sidebar appears in the left panel
with st.sidebar:
    st.header("Filters")
    category_filter = st.multiselect("Category", ["Food", "Transport"])
    date_range = st.date_input("Date range", value=(datetime.date(2026,1,1), datetime.date.today()))

# Alternative dot syntax (same result)
min_amount = st.sidebar.number_input("Min amount (Â£)", value=0.0)

### 4d. Expander â€” collapsible section

In [ ]:
with st.expander("Show raw data", expanded=False):
    st.write("Hidden until user clicks")

# Container â€” groups elements, useful for dynamic content
box = st.container(border=True)  # border=True draws a card outline
box.write("Inside a card")

---
## 5. Displaying Data

### 5a. DataFrames and Tables

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Date': ['2026-06-01', '2026-06-02', '2026-06-03'],
    'Description': ['Groceries', 'Claude.ai', 'Transport'],
    'Amount': [-45.20, -18.00, -3.50],
    'Category': ['Food', 'Subscriptions', 'Transport']
})

# Basic display â€” sortable, scrollable, read-only
st.dataframe(df, use_container_width=True, hide_index=True)

# Column config â€” control how each column looks
st.dataframe(
    df,
    column_config={
        "Date": st.column_config.DateColumn("Date", format="DD/MM/YYYY"),
        "Amount": st.column_config.NumberColumn("Amount (Â£)", format="%.2f"),
        "Category": st.column_config.TextColumn("Category"),
    },
    use_container_width=True,
    hide_index=True
)

# Editable table â€” user can change values, returns edited copy
edited = st.data_editor(
    df,
    column_config={
        "Category": st.column_config.SelectboxColumn(
            "Category",
            options=["Food", "Transport", "Subscriptions", "Uncategorized"]
        )
    },
    use_container_width=True,
    hide_index=True,
    key="my_editor"  # key is required when using data_editor
)

---
## 6. Charts

### 6a. Built-in quick charts

In [ ]:
import pandas as pd

spending = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr'],
    'Amount': [320, 410, 280, 390]
})

st.line_chart(spending.set_index('Month'))   # line chart
st.bar_chart(spending.set_index('Month'))    # bar chart
st.area_chart(spending.set_index('Month'))   # area chart

### 6b. Plotly â€” full control (recommended for personal projects)

In [ ]:
import plotly.express as px

df = pd.DataFrame({
    'Category': ['Food', 'Transport', 'Subscriptions', 'Other'],
    'Amount': [150, 60, 35, 95]
})

# Pie chart
fig = px.pie(df, values='Amount', names='Category', title='Spending by Category')
st.plotly_chart(fig, use_container_width=True)

# Bar chart
fig2 = px.bar(df, x='Category', y='Amount', title='Expenses', color='Category')
st.plotly_chart(fig2, use_container_width=True)

# Line chart with multiple series
monthly = pd.DataFrame({
    'Month': ['Jan','Feb','Mar','Apr'],
    'Income': [1200, 1200, 1350, 1200],
    'Expenses': [800, 950, 720, 880]
})
fig3 = px.line(monthly, x='Month', y=['Income','Expenses'], title='Income vs Expenses')
st.plotly_chart(fig3, use_container_width=True)

---
## 7. Session State

Because the script reruns on every interaction, normal variables reset.
`st.session_state` persists across reruns within the same browser session.

Think of it as a dictionary that survives reruns.

In [ ]:
# --- Basic usage ---

# Initialise once (guard prevents overwriting on every rerun)
if 'count' not in st.session_state:
    st.session_state.count = 0

if st.button("Increment"):
    st.session_state.count += 1

st.write(f"Count: {st.session_state.count}")

# --- Storing a DataFrame across reruns ---
if 'df' not in st.session_state:
    st.session_state.df = None

uploaded = st.file_uploader("Upload CSV")
if uploaded:
    st.session_state.df = pd.read_csv(uploaded)  # stored â€” survives reruns

if st.session_state.df is not None:
    st.dataframe(st.session_state.df)

# --- Force a rerun manually ---
# Useful after you modify session state and want the UI to reflect it immediately
if st.button("Reset"):
    st.session_state.count = 0
    st.rerun()  # triggers immediate rerun

# --- Key rule ---
# Always initialise with `if 'key' not in st.session_state` before reading,
# otherwise you'll get a KeyError on first load.

---
## 8. Forms

Without a form, every widget interaction reruns the script immediately.
With a form, nothing runs until the user clicks the submit button â€” good for multi-field inputs.

In [ ]:
with st.form("add_transaction"):
    st.subheader("Add a manual transaction")
    
    col1, col2 = st.columns(2)
    desc = col1.text_input("Description")
    amount = col2.number_input("Amount (Â£)", step=0.01)
    category = st.selectbox("Category", ["Food", "Transport", "Subscriptions"])
    date = st.date_input("Date")
    
    submitted = st.form_submit_button("Add", type="primary")

# Logic runs only when submitted is True
if submitted:
    if not desc:
        st.error("Description is required")
    else:
        st.success(f"Added: {desc} â€” Â£{amount:.2f} ({category})")

---
## 9. Caching

Caching prevents expensive operations (reading files, API calls, heavy computation) from re-running on every rerun.

- `@st.cache_data` â€” for data (DataFrames, lists, dicts). Returns a copy each time.
- `@st.cache_resource` â€” for shared resources (database connections, ML models). Returns the same object.

In [ ]:
# Without cache: reads the file on EVERY rerun
# With cache: reads once, returns cached result on subsequent reruns

@st.cache_data
def load_csv(file):
    return pd.read_csv(file)

@st.cache_data
def expensive_calculation(n):
    # Cached per unique value of n
    return sum(range(n))

# Clear cache for a specific function
# load_csv.clear()

# TTL â€” auto-expire cache after N seconds
@st.cache_data(ttl=300)  # 5 minutes
def fetch_live_data():
    pass  # e.g. API call

# Shared resource â€” same object returned every time (not a copy)
@st.cache_resource
def get_db_connection():
    import sqlite3
    return sqlite3.connect('finance.db', check_same_thread=False)

---
## 10. File Upload & Download

In [ ]:
# --- Upload ---
file = st.file_uploader(
    "Upload your CSV",
    type=['csv', 'xlsx'],          # restrict allowed types
    accept_multiple_files=False    # set True to allow multiple
)

if file is not None:
    df = pd.read_csv(file)
    st.dataframe(df)

# --- Download ---
# Convert DataFrame to CSV bytes, then offer as download
if 'df' in st.session_state and st.session_state.df is not None:
    csv_bytes = st.session_state.df.to_csv(index=False).encode('utf-8')
    
    st.download_button(
        label="Download as CSV",
        data=csv_bytes,
        file_name="transactions.csv",
        mime="text/csv"
    )

---
## 11. Status & Feedback

In [ ]:
# Spinner â€” shows while code runs
with st.spinner("Loading..."):
    import time
    time.sleep(1)  # simulate slow operation
st.success("Done!")

# Progress bar
import time
bar = st.progress(0, text="Processing...")
for i in range(100):
    bar.progress(i + 1, text=f"Processing {i+1}%")
    time.sleep(0.01)
bar.empty()  # remove bar when done

# Toast â€” small temporary notification (top-right corner)
st.toast("Changes saved!", icon="âœ…")

# Balloons / snow â€” fun celebration
st.balloons()
# st.snow()

---
## 12. Putting It All Together â€” Mini Finance App

This is a condensed but complete working example that uses most of the above concepts.

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import datetime

st.set_page_config(page_title="Finance App", page_icon="ðŸ’°", layout="wide")
st.title("Personal Finance Dashboard")

# --- Session state init ---
if 'df' not in st.session_state:
    st.session_state.df = None

# --- Sidebar filters ---
with st.sidebar:
    st.header("Filters")
    uploaded = st.file_uploader("Upload CSV", type="csv")
    if uploaded:
        df_raw = pd.read_csv(uploaded)
        df_raw['Date'] = pd.to_datetime(df_raw['Date'])
        st.session_state.df = df_raw

    if st.session_state.df is not None:
        categories = st.session_state.df['Category'].unique().tolist()
        selected_cats = st.multiselect("Categories", categories, default=categories)

# --- Main content ---
if st.session_state.df is None:
    st.info("Upload a CSV file to get started.")
    st.stop()  # st.stop() halts execution here â€” nothing below runs

df = st.session_state.df
df = df[df['Category'].isin(selected_cats)]

# --- KPI row ---
total_in = df[df['Amount'] > 0]['Amount'].sum()
total_out = df[df['Amount'] < 0]['Amount'].sum()
net = total_in + total_out

c1, c2, c3 = st.columns(3)
c1.metric("Income", f"Â£{total_in:.2f}")
c2.metric("Expenses", f"Â£{abs(total_out):.2f}")
c3.metric("Net", f"Â£{net:.2f}", delta=f"Â£{net:.2f}")

# --- Tabs ---
tab1, tab2 = st.tabs(["Transactions", "Charts"])

with tab1:
    st.dataframe(
        df,
        use_container_width=True,
        hide_index=True,
        column_config={
            "Amount": st.column_config.NumberColumn("Amount", format="%.2f GBP")
        }
    )
    csv = df.to_csv(index=False).encode()
    st.download_button("Download filtered CSV", csv, "filtered.csv", "text/csv")

with tab2:
    outflow = df[df['Amount'] < 0].copy()
    outflow['Amount'] = outflow['Amount'].abs()
    totals = outflow.groupby('Category')['Amount'].sum().reset_index()

    col1, col2 = st.columns(2)
    with col1:
        fig = px.pie(totals, values='Amount', names='Category', title='By Category')
        st.plotly_chart(fig, use_container_width=True)
    with col2:
        fig2 = px.bar(totals, x='Category', y='Amount', title='Spending Bar Chart')
        st.plotly_chart(fig2, use_container_width=True)

---
## 13. Pandas Patterns Used in the Finance App

These are the specific pandas operations introduced when building the combined transactions table and expense summary.

### 13a. `pd.concat()` — Combining Two DataFrames

Stacks two DataFrames on top of each other (by rows). Used when you have split data (e.g. inflows and outflows) that you want to display together.

In [ ]:
import pandas as pd

outflow_df = pd.DataFrame({
    'Date': ['2026-06-01', '2026-06-02'],
    'Description': ['Groceries', 'Claude.ai'],
    'Amount': [-45.20, -18.00]
})

inflow_df = pd.DataFrame({
    'Date': ['2026-06-01'],
    'Description': ['Bank Transfer'],
    'Amount': [200.00]
})

# Combine both, sort by date, reset index to 0,1,2...
combined = pd.concat([outflow_df, inflow_df]).sort_values('Date').reset_index(drop=True)
print(combined)

### 13b. `.apply(lambda x: ...)` — Transforming a Column Row by Row

Runs a function on every value in a column. The `lambda` is a short one-line function.

Used to split the single `Amount` column into separate `Outflows` and `Inflows` columns.

In [ ]:
# lambda x: ... means 'for each value x in this column, return...'
combined['Outflows'] = combined['Amount'].apply(lambda x: abs(x) if x < 0 else 0.0)
# if amount is negative (payment) -> absolute value  e.g. -45.20 becomes 45.20
# if amount is positive (inflow)  -> 0.0

combined['Inflows'] = combined['Amount'].apply(lambda x: x if x > 0 else 0.0)
# if amount is positive (inflow)  -> keep it
# if amount is negative (payment) -> 0.0

combined['Expenses'] = combined['Outflows'] - combined['Inflows']

print(combined[['Amount', 'Outflows', 'Inflows', 'Expenses']])

### 13c. `.groupby().sum()` — Totalling by Category

Groups rows that share the same value and sums a column. Used for the expense summary table.

In [ ]:
df = pd.DataFrame({
    'Category': ['Food', 'Food', 'Transport', 'Subscriptions'],
    'Amount': [-45.20, -12.00, -3.50, -18.00]
})

# Group by Category, sum Amount, reset_index gives a normal DataFrame back
totals = df.groupby('Category')['Amount'].sum().reset_index()
print(totals)

# .abs() converts all values to positive (payments are negative in the CSV)
totals['Amount'] = totals['Amount'].abs()

# Or chain it in one line:
totals2 = df.groupby('Category')['Amount'].sum().abs().reset_index()

### 13d. `.merge()` — Joining Two DataFrames on a Shared Column

Like a SQL JOIN. Used to combine the outflow totals and inflow totals into one summary table by matching on `Category`.

In [ ]:
outflow_totals = pd.DataFrame({
    'Category': ['Food', 'Transport', 'Subscriptions'],
    'Outflows': [57.20, 3.50, 18.00]
})

inflow_totals = pd.DataFrame({
    'Category': ['Salary', 'Food'],   # not all categories match
    'Inflows': [1200.00, 10.00]
})

# how='outer' keeps ALL categories from both tables
# missing values become NaN -- .fillna(0) replaces them with 0
merged = outflow_totals.merge(inflow_totals, on='Category', how='outer').fillna(0)
print(merged)

merged['Expenses'] = merged['Outflows'] - merged['Inflows']

### 13e. Appending a Total Row

Builds a summary footer and concatenates it to the bottom of the display DataFrame.

In [ ]:
display_df = pd.DataFrame({
    'Date': ['01/06/2026', '02/06/2026'],
    'Description': ['Groceries', 'Claude.ai'],
    'Outflows': [45.20, 18.00],
    'Inflows': [0.0, 0.0],
    'Expenses': [45.20, 18.00]
})

# Build the total row as a single-row DataFrame
total_row = pd.DataFrame([{
    'Date': '',
    'Description': 'TOTAL',
    'Outflows': display_df['Outflows'].sum(),
    'Inflows': display_df['Inflows'].sum(),
    'Expenses': display_df['Expenses'].sum()
}])

# ignore_index=True gives a clean continuous index after appending
display_with_total = pd.concat([display_df, total_row], ignore_index=True)
print(display_with_total)

### 13f. `.strftime()` — Formatting Dates as Strings

Converts a date object into a formatted string. Needed when mixing date objects and empty strings in the same column (the total row has an empty date), since `DateColumn` config only works with actual dates.

In [ ]:
import datetime

d = datetime.date(2026, 6, 1)

d.strftime('%d/%m/%Y')   # -> '01/06/2026'
d.strftime('%Y-%m-%d')   # -> '2026-06-01'
d.strftime('%B %Y')      # -> 'June 2026'

# Apply to a whole column:
# display_df['Date'] = display_df['Date'].apply(lambda x: x.strftime('%d/%m/%Y'))
# Converts every date object in the column to a string

### 13g. Widget `key=` Parameter — Avoiding Conflicts

When the same widget type appears more than once in a script (e.g. two buttons or two data editors), Streamlit needs a unique `key` for each. Without it you get a `DuplicateWidgetID` error.

In [ ]:
# Two buttons with the same label -- needs unique keys
if st.button('Apply Changes', type='primary', key='save_outflow'):
    pass  # handle outflow save

if st.button('Apply Changes', type='primary', key='save_inflow'):
    pass  # handle inflow save

# Two data editors -- must have different keys
edited_outflow = st.data_editor(outflow_df, key='outflow_editor')
edited_inflow  = st.data_editor(inflow_df,  key='inflow_editor')

# Rule: any widget that appears more than once in a script needs a unique key

### 13h. Filtering a DataFrame Before Passing to a Chart

Remove rows you don't want plotted before handing the data to Plotly. A pie chart cannot handle negative values — it silently fails to render.

In [ ]:
category_totals = pd.DataFrame({
    'Category': ['Food', 'Transport', 'Salary'],
    'Expenses': [57.20, 3.50, -1200.00]   # Salary is negative (net inflow)
})

import plotly.express as px

# Filter BEFORE passing to pie chart -- only plot categories where Expenses > 0
fig = px.pie(
    category_totals[category_totals['Expenses'] > 0],
    values='Expenses',
    names='Category',
    title='Expenses by Category'
)
st.plotly_chart(fig, use_container_width=True)

---
## Updated Cheat Sheet — Pandas Patterns

| Task | Code |
|------|------|
| Combine two DataFrames | `pd.concat([df1, df2]).reset_index(drop=True)` |
| Sort by column | `.sort_values('Date')` |
| Transform column values | `df['col'].apply(lambda x: ...)` |
| Absolute value of column | `df['col'].abs()` |
| Group and sum | `df.groupby('Category')['Amount'].sum().reset_index()` |
| Join two DataFrames | `df1.merge(df2, on='Category', how='outer')` |
| Replace NaN with 0 | `.fillna(0)` |
| Append a row | `pd.concat([df, new_row], ignore_index=True)` |
| Format date as string | `date.strftime('%d/%m/%Y')` |
| Format column of dates | `df['Date'].apply(lambda x: x.strftime('%d/%m/%Y'))` |
| Filter rows | `df[df['Amount'] > 0]` |
| Unique widget key | `st.button('Label', key='unique_key')` |

---
## Quick Reference Cheat Sheet

| Task | Code |
|------|------|
| Page config | `st.set_page_config(page_title=..., layout='wide')` |
| Title / header | `st.title()`, `st.header()`, `st.subheader()` |
| KPI card | `st.metric(label, value, delta)` |
| Text input | `val = st.text_input("Label")` |
| Number input | `val = st.number_input("Label", min_value=0)` |
| Dropdown | `val = st.selectbox("Label", options)` |
| Multi-select | `vals = st.multiselect("Label", options)` |
| Button | `if st.button("Label"): ...` |
| Columns | `c1, c2 = st.columns(2)` |
| Tabs | `t1, t2 = st.tabs(["A", "B"])` |
| Sidebar | `with st.sidebar:` |
| Expander | `with st.expander("Label"):` |
| Show DataFrame | `st.dataframe(df, use_container_width=True)` |
| Editable table | `edited = st.data_editor(df, key="k")` |
| Plotly chart | `st.plotly_chart(fig, use_container_width=True)` |
| Session state | `if 'x' not in st.session_state: st.session_state.x = ...` |
| Force rerun | `st.rerun()` |
| Stop execution | `st.stop()` |
| Cache function | `@st.cache_data` |
| File upload | `f = st.file_uploader("Label", type=['csv'])` |
| File download | `st.download_button("Label", data, filename)` |
| Spinner | `with st.spinner("..."):` |
| Toast | `st.toast("Message", icon="âœ…")` |
| Form | `with st.form("key"): ... st.form_submit_button()` |